# Building a training dataset from an orthomosaic

Three inputs go in — an orthomosaic, a stem shapefile, an AOI polygon — and a
`train/` + `mask/` folder pair comes out, ready for `training.run_train`.

The whole thing is three steps, and the order matters:

| | step | why it must come here |
|---|---|---|
| 1 | **fix** the geometry | an invalid ring aborts GEOS mid-run, after hours of sampling |
| 2 | **sample** tiles | needs valid geometry to intersect against |
| 3 | **split** train/val/test | must be spatial; a random split of oversampled tiles leaks |

`scripts/make_splits.py` runs all three in that order for a whole corpus. This notebook
does one site by hand so the arithmetic in the middle is visible.

In [ ]:
import os, sys, math, json
sys.path.insert(0, "..")            # run from notebooks/

GIS = "/Users/christian/data/Winmol/training_data/WINMOL_Trainings_Data/GIS"
SITE = "20210706_EW_WW_Kaufland"     # beech, the finest-resolution beech ortho we have

ORTHO = f"{GIS}/Orthomosaics/beech/{SITE}_ortho.tif"
STEMS = f"{GIS}/Training_Data/{SITE}.shp"      # the digitised stem polygons
AOI   = f"{GIS}/Polygone/{SITE}_AOE.shp"       # the windthrow area actually surveyed
OUT   = os.path.expanduser("~/winmol_data/demo")

for p in (ORTHO, STEMS, AOI):
    print(f"{'ok ' if os.path.exists(p) else 'MISSING'} {p}")

## What is actually on disk

Look before sampling. Two numbers decide everything downstream: the orthomosaic's ground
sample distance, and how much AOI survives the inward buffer.

In [ ]:
import fiona, rasterio
from shapely.geometry import shape
from shapely.ops import unary_union

with rasterio.open(ORTHO) as r:
    gsd, crs, wh = r.res[0], r.crs, (r.width, r.height)
with fiona.open(AOI) as c:
    aoi = unary_union([shape(f["geometry"]).buffer(0) for f in c])
with fiona.open(STEMS) as c:
    stems = [shape(f["geometry"]) for f in c]

print(f"ortho     {wh[0]} x {wh[1]} px, {gsd*100:.2f} cm/px, {crs}")
print(f"AOI       {aoi.area:,.0f} m²")
print(f"stems     {len(stems)} polygons, {sum(s.area for s in stems):,.0f} m² of stem")
print(f"invalid   {sum(not s.is_valid for s in stems)} polygons")

## The scale arithmetic

A tile is a square of `extent_m` metres on the ground, resampled to `tile_px` pixels. So

> **effective GSD = `extent_m / tile_px`**

and this is the *only* thing the model ever sees — the orthomosaic's own resolution drops
out. Match it to how the Analyzer will serve the model: `ExecutionPlan.py:95-97` cuts
`ceil(tile_size / pixel_size)` source pixels and resizes to 512, giving
`tile_size / 512`. At the default `tile_size=15` that is **2.93 cm/px**.

Tiles are also rotated by a random angle, so the footprint must stay inside the AOI at
*any* rotation. That needs an inward buffer of the square's half-diagonal,
`extent_m · √2 / 2`. Anything less and tiles hang over the edge into undigitised ground,
where real stems count as false negatives.

In [ ]:
print(f"{'extent':>7} {'tile':>6} {'eff GSD':>9} {'resample':>10} {'buffer':>7} {'usable AOI':>11}")
for extent_m in (10.24, 15.0, 20.0):
    tile_px = 512
    eff = extent_m / tile_px
    factor = (extent_m / gsd) / tile_px      # >1 downsamples, <1 upsamples
    buf = extent_m * math.sqrt(2) / 2
    usable = aoi.buffer(-buf).area
    tag = "DOWN" if factor > 1.02 else ("up" if factor < 0.98 else "1:1")
    print(f"{extent_m:7.2f} {tile_px:6d} {eff*100:8.2f}cm {factor:6.2f} {tag:<4}"
          f" {buf:6.2f}m {usable:10,.0f}m²")

Read that table before choosing `extent`. It tells you two things at once:

- **whether tiles are up- or downsampled.** Only downsampling loses information, and only
  downsampling makes anti-aliasing matter at all.
- **how much ground you actually get.** A larger extent needs a larger buffer, and on a
  small AOI the buffer can eat the site. That is why the native-1024 experiment was
  impossible on this corpus, not a bug.

## Step 1 — fix the geometry

`make_valid` on self-intersecting rings. Cheap, and it turns a mid-run GEOS abort into a
line of output. The corpus had 50 invalid polygons across 8 sites.

In [ ]:
from scripts.fix_geometries import fix

os.makedirs(OUT, exist_ok=True)
STEMS_FIXED = f"{OUT}/{SITE}_fixed.shp"
rep = fix(STEMS, out_path=STEMS_FIXED, report_path=f"{OUT}/repairs.json", quiet=True)

# `entries` holds one record per repair; keep the summary here, it is in the JSON
{k: v for k, v in rep.items() if k not in ("entries", "dropped_entries")}

## Step 2 — sample tiles

For each attempt: draw a random point in the buffered AOI, pick a random angle, cut the
rotated square, rasterise the stems that fall inside it, and keep the tile only if the
*finished mask* carries at least `min_stem_frac` stem pixels.

Measuring the finished mask — rather than the polygons that intersect the footprint — is
deliberate. The original R script summed the **whole** area of every polygon touching the
footprint, so tiles qualified on stems lying mostly outside them. That single rule costs
8–27 F1 points.

In [ ]:
from scripts.sample_training_tiles import sample_tiles

stats = sample_tiles(
    ortho=ORTHO, stems=STEMS_FIXED, aoi=AOI, out_dir=f"{OUT}/all",
    extent_m=10.24,        # -> 2.00 cm/px effective
    tile_px=512,
    oversample=25,         # attempts per footprint-sized cell; R used 100
    min_stem_frac=1/200,   # reject tiles with less stem than this
    seed=1,
    quiet=False,
)
stats

**Read the rejection counts, not just the written count.** A high `too_few_stems` means
the AOI is sparser than the floor assumes; a nonzero `nodata` means the footprint
is running off the imagery.

## Step 3 — split, spatially

**Never split these tiles randomly.** At 25× oversampling the footprints overlap heavily,
so a random split puts near-duplicate tiles on both sides and the validation score becomes
fiction.

Instead cut the AOI into blocks, assign whole blocks to splits, and sample each split
separately with the same `split_seed`. Re-run the cell below once per split.

In [ ]:
for split in ("train", "val"):
    sample_tiles(
        ortho=ORTHO, stems=STEMS_FIXED, aoi=AOI, out_dir=f"{OUT}/{split}",
        extent_m=10.24, tile_px=512, oversample=25, seed=1,
        block_size_m=50,
        split=split,
        split_fractions={"train": 0.7, "val": 0.3, "test": 0.0},
        split_seed=1,          # identical across splits, or blocks land on both sides
        quiet=True,
    )
    n = len(os.listdir(f"{OUT}/{split}/train"))
    print(f"{split:6} {n:5d} tiles")

## Verify the split is leak-free — numerically

Do not assert this, measure it. Every dataset carries `tiles.jsonl` with each tile's world
centre, so the closest train↔val approach can be computed directly and compared against
the distance below which two footprints can overlap at all.

In [ ]:
import itertools

def centres(split):
    with open(f"{OUT}/{split}/tiles.jsonl") as f:
        rows = [json.loads(l) for l in f]
    return [(r["centre_x"], r["centre_y"]) for r in rows], rows[0]["extent_m"]

tr, extent_m = centres("train")
va, _ = centres("val")

closest = min(math.dist(a, b) for a, b in itertools.product(tr, va))
threshold = extent_m * math.sqrt(2)          # two rotated squares can touch below this

print(f"closest train<->val centre distance : {closest:7.2f} m")
print(f"overlap threshold (extent * sqrt2)  : {threshold:7.2f} m")
print("LEAK-FREE" if closest > threshold else "*** TILES CAN OVERLAP — do not trust val ***")

## Look at the tiles

A random sample, seed stated. Not the best ones — a selected figure misrepresents, and
mask misalignment is only ever caught by eye.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

random.seed(7)
ids = random.sample(range(1, len(os.listdir(f"{OUT}/train/train")) + 1), 4)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, i in zip(axes, ids):
    rgb = np.asarray(Image.open(f"{OUT}/train/train/train{i}.jpeg"))
    msk = np.asarray(Image.open(f"{OUT}/train/mask/mask{i}.gif")) > 127
    ax.imshow(rgb)
    # outline, not fill: a fill hides the evidence you are checking
    ax.contour(msk, levels=[0.5], colors="#00E5FF", linewidths=1.1)
    ax.set_title(f"tile {i} — {msk.mean()*100:.1f}% stem", fontsize=9)
    ax.axis("off")
fig.suptitle("random sample, seed=7 — stem mask outlined", fontsize=10)
plt.tight_layout()

If a mask looks wrong, trace it back to the ground rather than guessing:

```bash
python scripts/locate_tile.py --data-dir ~/winmol_data/demo/train --tile 42 --crop out.png
```

That re-cuts the same footprint from the orthomosaic at native resolution, which usually
settles whether the label or the sampler is at fault.

## Train on it

```bash
python -m training.run_train --arch hrnet \
    --data-dir      ~/winmol_data/demo/train \
    --val-data-dir  ~/winmol_data/demo/val \
    --test-data-dir ~/winmol_data/demo/test \
    --epochs 30 --batch-size 16 --device cuda \
    --out-dir output/demo
```

`hrnet` at 16.1M from scratch is the strongest configuration measured on this corpus —
ahead of an 82M pretrained SegFormer and a 203M DINOv3 ConvNeXt. The run writes a
contract-conformant `model.onnx` the Analyzer can serve directly.

### For a real experiment, not a demo

- **Use several sites.** One site cannot tell segmentation quality from domain transfer;
  site identity has outweighed every other variable measured in this project.
- **Hold out a whole orthomosaic for test**, and check its validation signal is alive
  before trusting any checkpoint — a site that returns F1 0.0000 once voided six runs here.
- `scripts/make_splits.py --config … --strategy sites` does all of the above for a corpus.